# Llama3 Fine-Tuning Model Evaluation

## Import Libraries

In [1]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

In [2]:
!wget -q https://raw.githubusercontent.com/KumudithaSilva/llama3-domain-adaptation/feature-base-model/evaluator.py -O evaluator.py

In [15]:
import getpass
from huggingface_hub import login
from datasets import load_dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from evaluator import evaluate

## Load Dataset From HuggingFace

In [16]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"

PROJECT_NAME = "stream_price"
TASK = "fine-tuning"

DATA_USER = "KumudithaSilva"
DATASET_NAME = f"{DATA_USER}/stream_items_prompt_lite"

RUN_NAME =  f"{TASK}-{"2026-04-30_11.01.12"}"

FINE_TUNED_MODEL_HUGGINGFACE = "de4a39b1ae7666620aa49a6bff90a97190b098fd"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{DATA_USER}/{PROJECT_RUN_NAME}"

In [5]:
print(f"DATASET_NAME: {DATASET_NAME}")
print(f"RUN_NAME: {RUN_NAME} \n")
print(f"PROJECT_RUN_NAME: {PROJECT_RUN_NAME}")
print(f"HUB_MODEL_NAME: {HUB_MODEL_NAME}")

## Log in to HuggingFace

In [8]:
hf_token = getpass.getpass("Enter Huggingface Key:")
login(hf_token)

## Load Test Dataset From HuggingFace


In [9]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test'].remove_columns(['id'])

## Load Llama Model

### Quantization

In [11]:
# Quantization config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
    )

### Tokenizer

In [12]:
# Tokenizer config
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

### Base Model

In [13]:
# Base model config
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    )

base_model.generation_config.pad_token_id = tokenizer.pad_token_id

## Fine-Tuned Model

In [17]:
fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=FINE_TUNED_MODEL_HUGGINGFACE, device_map="auto")

In [18]:
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

## Model Prediction

In [21]:
def model_predict(item):
    inputs = tokenizer(item["prompt"],return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]

    return tokenizer.decode(generated_ids)

In [22]:
evaluate(model_predict, test)